<a href="https://colab.research.google.com/github/ayeshasiddiqa2891999-blip/Careloop-noshow-guard/blob/main/capestone_project1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!ls

appointments.csv  clinics.csv  followup_notes.csv  sample_data


In [ ]:
#task 1
import csv
with open("appointments.csv",newline="") as f:
  reader = csv.DictReader(f)
  rows = list(reader)
print(reader.fieldnames)
print(len(rows))

['appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date', 'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel', 'visit_type', 'reminder_sent', 'prior_no_shows', 'status']
1020


In [ ]:
import pandas as pd
df=pd.read_csv("appointments.csv")
print(df.shape)
df.info()
print(df.isnull().sum())
print(df.duplicated().sum())

df=df.drop_duplicates()
print(len(df))
df["booking_channel"]= df["booking_channel"].str.strip().str.title()
df["appt_date"]= pd.to_datetime(df["appt_date"],errors="coerce")
df["booked_on"]= pd.to_datetime(df["booked_on"],errors="coerce")
df= df.dropna(subset=["appt_date","status"])
print(len(df))

(1020, 14)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1020 entries, 0 to 1019
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   appointment_id   1020 non-null   object 
 1   patient_id       1020 non-null   object 
 2   clinic_id        1020 non-null   object 
 3   booked_on        1020 non-null   object 
 4   appt_date        1019 non-null   object 
 5   age              1008 non-null   float64
 6   gender           1020 non-null   object 
 7   distance_km      1000 non-null   float64
 8   fee_inr          1020 non-null   int64  
 9   booking_channel  1020 non-null   object 
 10  visit_type       1020 non-null   object 
 11  reminder_sent    1020 non-null   object 
 12  prior_no_shows   1020 non-null   int64  
 13  status           1016 non-null   object 
dtypes: float64(2), int64(2), object(10)
memory usage: 111.7+ KB
appointment_id      0
patient_id          0
clinic_id           0
booked_on      

In [ ]:
import numpy as np
#bad values --->missing
print(((df["age"]<1)|(df["age"]>100).sum()))
df.loc[(df["age"]<1)|(df["age"]>100),"age"]=np.nan
print(df["age"].median())

# fill gaps with median
df["age"]= df["age"].fillna(df["age"].median()).round().astype(int)
df["distance_km"] = df["distance_km"].fillna(df["distance_km"].median())
print(df["age"].median(), df["distance_km"].median())

# outliers:Q1,Q3,IQR,fences,cap
q1= df["distance_km"].quantile(0.25)
q3 = df["distance_km"].quantile(0.75)
iqr =  q3-q1
low , high = q1-1.5*iqr,q3+1.5*iqr
print((df["distance_km"] < low).sum() + (df["distance_km"] > high).sum())
df["distance_km"]= np.clip(df["distance_km"], low,high)

#Outliers: for fee_inr
q1= df["fee_inr"].quantile(0.25)
q3 = df["fee_inr"].quantile(0.75)
iqr =  q3-q1
low , high = q1-1.5*iqr,q3+1.5*iqr
print((df["fee_inr"] < low).sum() + (df["fee_inr"] > high).sum())
df["fee_inr"]= np.clip(df["fee_inr"], low,high)


#rounding
df["distance_km"] = df["distance_km"].round(1)
df["fee_inr"]  = df["fee_inr"].round().astype(int)

0       True
1       True
2       True
4       True
5       True
        ... 
1015    True
1016    True
1017    True
1018    True
1019    True
Name: age, Length: 990, dtype: bool
48.0
48.0 8.25
26
3


In [ ]:
df["lead_days"] = (df["appt_date"]- df["booked_on"]).dt.days
df["no_show"] = (df["status"]=="No-Show").astype(int)
print(df["no_show"].sum())

mask= df["lead_days"]>14
print(mask.sum(),mask.mean())

x= df[["age","distance_km","fee_inr","lead_days"]].to_numpy()
print(np.mean(x, axis=0))
z=(x[:,1]- np.mean(x[:,1]))/np.std(x[:,1])
print((np.abs(z)>2).sum())


218
296 0.298989898989899
[ 48.17373737   9.50090909 635.71414141  11.81919192]
53


In [ ]:
print(df["status"].value_counts())

status
Attended    772
No-Show     218
Name: count, dtype: int64


In [ ]:
import requests
import pandas as pd

# Load dataset and prepare a list for temperatures
clinics = pd.read_csv("clinics.csv")
temps = []

for _, row in clinics.iterrows():
    temp_c = None
    for attempt in range(2):
        try:
            r = requests.get(
                "https://api.open-meteo.com/v1/forecast",
                params={
                    "latitude": row["latitude"],
                    "longitude": row["longitude"],
                    "hourly": "temperature_2m",
                    "forecast_days": 1
                },
                timeout=10
            )
            if r.status_code == 200:
                temp_c = r.json()["hourly"]["temperature_2m"][0]
                break
        except requests.exceptions.RequestException as e:
            print(row["clinic_name"], e)
    temps.append(temp_c)

# Assign columns and save only after compiling all results in the loop
clinics["temp_c"] = temps
clinics.to_csv("clinics_enriched.csv", index=False)
print(clinics)

  clinic_id               clinic_name     city    tier  latitude  longitude  \
0       C01     Kothrud Family Clinic     Pune   Metro   18.5074    73.8077   
1       C02      Hadapsar Care Centre     Pune   Metro   18.5089    73.9260   
2       C03    Sitabuldi Health Point   Nagpur  Tier-2   21.1458    79.0882   
3       C04      Malviya Nagar Clinic   Jaipur  Tier-2   26.8553    75.8150   
4       C05     Vaishali Nagar Clinic   Jaipur  Tier-2   26.9124    75.7873   
5       C06  Edappally Medical Centre    Kochi  Tier-2   10.0261    76.3085   
6       C07        Vijay Nagar Clinic   Indore  Tier-2   22.7533    75.8937   
7       C08    Gomti Nagar Care Point  Lucknow  Tier-2   26.8467    80.9462   

   temp_c  
0    18.3  
1    20.0  
2    20.4  
3    16.2  
4    16.2  
5    26.2  
6    16.6  
7    16.9  


In [ ]:
#Task 1.6:
#Part A
df = df.sort_values("appointment_id")
df["booked_on"]= df["booked_on"].dt.strftime("%Y-%m-%d")
df["appt_date"] = df["appt_date"].dt.strftime("%Y-%m-%d")
print(df["appointment_id"].is_unique, df.isnull().sum().sum())

True 0


In [ ]:
#part b:
cols =["appointment_id","patient_id","clinic_id","booked_on", "appt_date","age","gender","distance_km", "fee_inr", "booking_channel",
       "visit_type","reminder_sent","prior_no_shows","status","lead_days","no_show"]
df[cols].to_csv("appointments_clean.csv",index=False)
print(df[cols].shape)

(990, 16)


In [ ]:
#part C:
with open ("load_data.sql","w") as f:
  for _,r in clinics.iterrows():
    temp= "NULL" if pd.isna(r["temp_c"]) else r["temp_c"]
    f.write(f"""INSERT INTO clinics VALUES('{r["clinic_id"]}','{r["clinic_name"]}','{r["city"]}','{r["tier"]}',
    {r["latitude"]},{r["longitude"]},{temp});\n""")

  for _,r in df[cols].iterrows():
      f.write(f"""INSERT INTO appointments VALUES('{r["appointment_id"]}','{r["patient_id"]}','{r["clinic_id"]}',
      '{r["booked_on"]}','{r["appt_date"]}',{r["age"]},{r["distance_km"]},{r["fee_inr"]},'{r["gender"]}','{r["booking_channel"]}',
      '{r["visit_type"]}','{r["reminder_sent"]}',{r["prior_no_shows"]},'{r["status"]}',{r["lead_days"]},{r["no_show"]});\n""")

In [ ]:
!head -3 load_data.sql

INSERT INTO clinics VALUES('C01','Kothrud Family Clinic','Pune','Metro',
    18.5074,73.8077,18.3);
INSERT INTO clinics VALUES('C02','Hadapsar Care Centre','Pune','Metro',


In [ ]:
!grep -c INSERT load_data.sql

998


In [ ]:
!grep -n 'Yes' load_data.sql | head -3

19:      'Consultation','Yes',1,'No-Show',1,1);
28:      'Consultation','Yes',0,'Attended',4,0);
34:      'Follow-up','Yes',0,'Attended',10,0);


In [ ]:
check = df.groupby("clinic_id").agg(
    orders= ("no_show","size"),
    returns = ("no_show","sum")
)
print(check)

           orders  returns
clinic_id                 
C01           160       36
C02           143       39
C03            82       13
C04           142       38
C05           130       18
C06           142       23
C07           101       21
C08            90       30


In [ ]:
#TASK 3--> TASK 3.1:
import pandas as pd
from sklearn.model_selection import train_test_split
data = pd.read_csv("appointments_clean.csv")
data["reminder_sent"] = data["reminder_sent"].map({"Yes": 1,"No": 0})
X= data[["age","distance_km","fee_inr","lead_days","prior_no_shows","reminder_sent","booking_channel","visit_type"]]
X= pd.get_dummies(X, columns=["booking_channel","visit_type"])
y= data["no_show"]
X_train, X_test, y_train, y_test = train_test_split(
    X,y,test_size=0.2 , random_state = 42
)
print(len(X_train), len(X_test),y_test.sum())
print(1-y_test.mean())

792 198 49
0.7525252525252525


In [ ]:
#Task 3.2: logistic regression
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, confusion_matrix, precision_score, recall_score, f1_score, roc_auc_score)
import numpy as np

scaler = StandardScaler()
Xtr_s= scaler.fit_transform(X_train)  #fit on train only
Xte_s = scaler.transform(X_test)

lr = LogisticRegression(max_iter = 1000).fit(Xtr_s, y_train)
pred = lr.predict(Xte_s)
prob = lr.predict_proba(Xte_s)[:,1]
print(accuracy_score(y_test,pred), precision_score(y_test, pred), recall_score(y_test,pred),f1_score(y_test,pred),
      roc_auc_score(y_test,prob))
print(confusion_matrix(y_test,pred))

coef = lr.coef_[0]
for name, c in zip(X.columns,coef): print(name,round(c,2))
i = list(X.columns).index("reminder_sent")
print(np.exp(coef[i]), np.exp(coef[i]/scaler.scale_[i]))

0.7878787878787878 0.6296296296296297 0.3469387755102041 0.4473684210526316 0.788933022873579
[[139  10]
 [ 32  17]]
age -0.25
distance_km 0.25
fee_inr 0.45
lead_days 0.61
prior_no_shows 0.65
reminder_sent -0.58
booking_channel_App -0.05
booking_channel_Phone 0.22
booking_channel_Walk-In -0.19
visit_type_Consultation 0.02
visit_type_Follow-up -0.02
0.559524141750829 0.3093980903124433


In [ ]:
print(roc_auc_score(y_test,prob))

0.788933022873579


In [ ]:
#Task 3.3: decision tree andoverfitting
from sklearn.tree import DecisionTreeClassifier
for d in [None, 3]:
  t = DecisionTreeClassifier(max_depth=d, random_state= 42).fit(X_train, y_train)
  print(d, t.score(X_train,y_train),t.score(X_test,y_test))


None 1.0 0.6717171717171717
3 0.821969696969697 0.7777777777777778


In [ ]:
from sklearn.tree import DecisionTreeClassifier
t3 = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X_train, y_train)
print(recall_score(y_test, t3.predict(X_test)),
      roc_auc_score(y_test, t3.predict_proba(X_test)[:, 1]))

0.22448979591836735 0.6996986714148747


In [ ]:
#Task 3.4: random forest and comparision table
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(n_estimators = 100, random_state=42).fit(X_train , y_train)
t3 = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X_train, y_train)
print(recall_score(y_test, t3.predict(X_test)),
      roc_auc_score(y_test, t3.predict_proba(X_test)[:, 1]))
p = rf.predict_proba(X_test)[:,1]
print(accuracy_score(y_test,rf.predict(X_test)),recall_score(y_test, rf.predict(X_test)),roc_auc_score(y_test,p))
print(sorted(zip(rf.feature_importances_,X.columns),reverse=True)[:,3])

0.22448979591836735 0.6996986714148747
0.7474747474747475 0.30612244897959184 0.7057252431173813


TypeError: list indices must be integers or slices, not tuple

In [ ]:
results = pd.DataFrame({
    "model": ["Logistic regression", "Tree depth 3", "Random forest"],
    "accuracy": [accuracy_score(y_test, pred),
                 accuracy_score(y_test, t3.predict(X_test)),
                 accuracy_score(y_test, rf.predict(X_test))],
    "recall":   [recall_score(y_test, pred),
                 recall_score(y_test, t3.predict(X_test)),
                 recall_score(y_test, rf.predict(X_test))],
    "auc":      [roc_auc_score(y_test, prob),
                 roc_auc_score(y_test, t3.predict_proba(X_test)[:, 1]),
                 roc_auc_score(y_test, p)]})
print(results.round(3))

                 model  accuracy  recall    auc
0  Logistic regression     0.788   0.347  0.789
1         Tree depth 3     0.778   0.224  0.700
2        Random forest     0.747   0.306  0.706


In [ ]:
print(accuracy_score(y_test,pred),
      recall_score(y_test,pred),
      roc_auc_score(y_test,prob))

0.7878787878787878 0.3469387755102041 0.788933022873579


In [ ]:
from sklearn.model_selection import GridSearchCV, cross_val_score, cross_val_predict
from sklearn.pipeline import Pipeline

# (a) tune the forest on TRAIN only
gs = GridSearchCV(RandomForestClassifier(random_state=42),
    {"n_estimators": [100, 200], "max_depth": [3, 5, 8]},
    cv=5, scoring="roc_auc").fit(X_train, y_train)
print(gs.best_params_, gs.best_score_)

# (b) compare by 5-fold CV AUC on train; use the unscaled X_train
lr_pipe = Pipeline([("scaler", StandardScaler()),
                    ("model", LogisticRegression(max_iter=1000))])
cv_auc = cross_val_score(lr_pipe, X_train, y_train, cv=5, scoring="roc_auc").mean()
# do the same for the depth-3 tree (cross_val_score) and gs.best_score_ for the forest
tree_auc = cross_val_score(DecisionTreeClassifier(max_depth=3, random_state=42),
                           X_train, y_train,cv=5, scoring="roc_auc").mean()
print("CV AUC logistic:", cv_auc,"tree:",tree_auc,"forest:",gs.best_score_)

# (c) pick the threshold with out-of-fold probabilities
oof = cross_val_predict(lr_pipe, X_train, y_train, cv=5, method="predict_proba")[:, 1]
for th in [0.5, 0.35]:
    f = oof >= th
    print(th, precision_score(y_train, f), recall_score(y_train, f),
          f1_score(y_train, f), f.sum())

# (d) fit the winner on train, evaluate on test ONCE
lr_pipe.fit(X_train, y_train)
probs = lr_pipe.predict_proba(X_test)[:, 1]
flag = (probs >= 0.35).astype(int)       # use YOUR chosen threshold
print(precision_score(y_test,flag), recall_score(y_test,flag),
      roc_auc_score(y_test,probs),flag.sum())

# (e) save the scores
out = pd.DataFrame({"appointment_id": data.loc[X_test.index, "appointment_id"],
                    "no_show": y_test, "risk_prob": probs, "flag": flag})
out.to_csv("test_risk_scores.csv", index=False)
print(out.shape, out["flag"].sum())

{'max_depth': 3, 'n_estimators': 200} 0.7673242251739405
CV AUC logistic: 0.7737139899948249 tree: 0.6784494968661952 forest: 0.7673242251739405
0.5 0.5753424657534246 0.2485207100591716 0.34710743801652894 73
0.35 0.512987012987013 0.46745562130177515 0.4891640866873065 154
0.5714285714285714 0.5714285714285714 0.788933022873579 49
(198, 4) 49


In [ ]:
# A clinic would choose the  threshold by how many follow-up calls the front desk make in evening against the cost of empty slots.